# 02: Transform an Existing Block Model Attribute

This tutorial signs in to Evo, selects a workspace during sign-in, loads an existing block model, transforms one numeric attribute, and publishes the result as a new attribute.

## Before You Start

You need an Evo app client ID and redirect URL. You also need permission to view and update the selected block model. The new attribute creates a new block model version; the source attribute is not changed.

## Learning Path

1. [01: Create and Query a Regular Block Model](01-create-and-query-regular-block-model.ipynb)
2. **This notebook**: transform an existing numeric attribute
3. [03: Block Model Reports](03-block-model-reports.ipynb)

In [ ]:
from evo.notebooks import ServiceManagerWidget

# Evo app credentials
client_id = "daves-evo-client"  # Replace with your client ID
redirect_url = "<your-redirect-url>"  # Replace with your redirect URL

# The sign-in flow includes workspace selection.
manager = await ServiceManagerWidget.with_auth_code(
    client_id=client_id,
    # redirect_url=redirect_url,
).login()

/Users/david.knight/Developer/forks/evo-python-sdk/code-samples/.venv/lib/python3.14/site-packages/evo/notebooks/authorizer.py:108: UserWarning: The evo.notebooks.AuthorizationCodeAuthorizer is not secure, and should only ever be used in Jupyter notebooks in a private environment.
  warnings.warn(


ServiceManagerWidget(children=(VBox(children=(HBox(children=(Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR…

SelectionError: No organization is currently selected.

## Select a Block Model

Choose a block model from the dropdown. The widget lists block models in the workspace selected during sign-in. After making a choice, run the next cell to load the model and list its attributes.

In [3]:
from evo.blockmodels import BlockModelAPIClient

from helpers import BlockModelSelectorWidget

environment = manager.get_environment()

service_client = BlockModelAPIClient(environment, manager.get_connector(), manager.cache)
block_model_selector = await BlockModelSelectorWidget.create(service_client)
display(block_model_selector)

BlockModelSelectorWidget(children=(Dropdown(description='Block model:', layout=Layout(width='auto'), options=(…

In [4]:
from evo.objects.data import ObjectReference
from evo.objects.typed import BlockModel

api_block_model = await service_client.get_block_model(block_model_selector.value)
object_id = api_block_model.geoscience_object_id
if object_id is None:
    raise RuntimeError("The selected block model is not linked to a geoscience object.")

object_reference = ObjectReference.new(environment, object_id=object_id)
block_model = await BlockModel.from_reference(manager, object_reference)

print(f"Selected block model: {block_model.name}")
print(f"Block model ID: {block_model.block_model_uuid}")
print("Available attributes:")
for attribute in block_model.attributes:
    print(f"- {attribute.name}")

Selected block model: Regular block model 2026-06-11 14:52:02.331802
Block model ID: 2a32be3b-30db-40dd-bd12-eba1232fdf1b
Available attributes:
- Geology
- Cu
- Ag
- Pt


## Choose and Transform a Numeric Attribute

Set `source_column` to one of the attributes listed above. This example applies $new = old \times scale + offset$. Keep the scale at `1.0` and the offset at `0.0` to make an exact copy. Choose a unique `new_column` name.

Use a unit compatible with the transformed values, or leave `new_column_unit` as `None` when no unit is required.

In [5]:
import pandas as pd

source_column = "Cu"  # Replace with an attribute name
new_column = f"{source_column}_transformed"
scale = 2.0
offset = 0.0
new_column_unit = None  # For example: "g/t"

if source_column in {"x", "y", "z"}:
    raise ValueError("Choose a block model attribute, not a geometry column.")
if block_model.get_attribute(source_column) is None:
    raise ValueError(f"{source_column!r} is not an attribute on the selected block model.")
if block_model.get_attribute(new_column) is not None:
    raise ValueError(f"{new_column!r} already exists. Choose a new attribute name.")

source_data = await block_model.to_dataframe(columns=["x", "y", "z", source_column])
if not pd.api.types.is_numeric_dtype(source_data[source_column]):
    raise TypeError(f"{source_column!r} must contain numeric values.")

new_attribute_data = source_data[["x", "y", "z"]].copy()
new_attribute_data[new_column] = source_data[source_column] * scale + offset

display(new_attribute_data.head())
print(f"Transform: {new_column} = {source_column} * {scale} + {offset}")

,x,y,z,Cu_transformed
0,1478912.5,5174512.5,612.5,0.020392
1,1478912.5,5174512.5,637.5,0.020400
2,1478912.5,5174512.5,662.5,0.020407
3,1478912.5,5174537.5,612.5,0.020417
4,1478912.5,5174537.5,637.5,0.020425


Transform: Cu_transformed = Cu * 2.0 + 0.0


## Publish the New Attribute

Run this cell only after checking the preview. Evo creates a new version containing the new attribute.

In [6]:
version = await block_model.add_attribute(
    data=new_attribute_data,
    attribute_name=new_column,
    unit=new_column_unit,
)

print(f"Published {new_column!r} in version: {version.version_id}")

Published 'Cu_transformed' in version: 4
